# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook guides you through exploring and processing the **A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya** using the `mlcroissant` library. The workflow includes loading the dataset, overviewing the schema, extracting tabular data, performing exploratory data analysis (EDA), and producing simple visualizations.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

We will load the dataset's Croissant schema and access its metadata. The `mlcroissant` library provides simple ways for exploring datasets and their structure following the [MLCommons Croissant](https://mlcommons.org/croissant/) metadata standard.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print high-level dataset metadata
md = dataset.metadata
print(f"Dataset Name: {md.name}\nDescription: {md.description}\nVersion: {getattr(md, 'version', 'N/A')}")

## 2. Data Overview

List available record sets, their fields, and their corresponding `@id`s. This helps identify relevant entity keys for later extraction. All references use Croissant `@id` as required.

In [ ]:
# Retrieve record sets from the dataset
record_sets = list(dataset.record_sets)
print("Available Record Sets (by @id):")
for recset in record_sets:
    print(f"- {recset.id}   ({recset.name})")

# For demonstration, let's show the first record set's fields and their IDs
if len(record_sets) > 0:
    main_record_set = record_sets[0]
    print(f"\nFields in Record Set '{main_record_set.name}' (id: {main_record_set.id}):")
    for field in main_record_set.fields:
        print(f"  - {field.id}  (name: {field.name}, type: {getattr(field, 'data_type', 'N/A')})")

## 3. Data Extraction

Extract records from one or more record sets and load into pandas DataFrames for analysis. All entities are referenced by their `@id` fields as per Croissant schema. Replace `main_records_set_id` with the `@id` of the chosen record set for your analysis if needed.

In [ ]:
# Choose record sets for extraction by their @id
record_sets_ids = [recset.id for recset in record_sets]
dataframes = {}

for recset in record_sets:
    # Reads all records into a pandas DataFrame
    df = pd.DataFrame(list(dataset.records(record_set=recset.id)))
    dataframes[recset.id] = df
    print(f"Loaded DataFrame for RecordSet '@id': {recset.id} (shape: {df.shape})")

# Display columns of the first record set
main_records_set_id = record_sets_ids[0]
print(f"\nColumns in DataFrame for RecordSet '@id': {main_records_set_id}")
print(dataframes[main_records_set_id].columns.tolist())
dataframes[main_records_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Filter, normalize, and group data using Croissant `@id`s. For demonstration, numeric fields commonly present in mental health surveys are used (e.g., total score columns for GAD-7, PHQ-9 or PSQ). Update variables to match field/column `@id`s for your target field and grouping as necessary.

In [ ]:
# Identify likely numeric and groupable fields (by @id)
df = dataframes[main_records_set_id]
numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
print("Numeric fields detected:", numeric_field_candidates)

# You may need to choose fields by their actual @id. Example: 'gad7_total_score', 'phq9_total_score', 'psq_total_score'
# Adjust as needed based on actual column names (which should match field @id)
numeric_field_id = numeric_field_candidates[0] if len(numeric_field_candidates) else None
if numeric_field_id:
    threshold = df[numeric_field_id].mean()  # Example: filter for above average scores
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.1f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a group field; 'village', 'gender', etc are common (refer to @id). Here, pick the first string/categorical field.
    group_field_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field}, showing mean {numeric_field_id}:")
        print(grouped_df.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization

Visualize distributions (e.g., histogram of total scores or box plots per group). Update field variables to match the dataset columns (by @id).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group field
    if group_field_candidates:
        group_field = group_field_candidates[0]
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated:
- Loading a Croissant-based dataset with `mlcroissant`.
- Identifying record sets and accessing fields by their `@id`.
- Extracting records as pandas DataFrames.
- Conducting exploratory processing and simple visualizations with field and record set references strictly by `@id`.

This workflow provides a robust starting point for in-depth data analysis, visualization, and machine learning using FAIR data standards in social science, health, or other structured survey datasets.